#SQL QUERIES ON CLEANED CUSTOMER DATA
# Purpose: Extract insights using SQL before dashboard
# Input: customer_segmentation_cleaned.csv
# Output: SQL results for Power BI

In [1]:
import pandas as pd
import sqlite3

In [2]:
print("="*60)
print("SQL ANALYSIS ON CUSTOMER SEGMENTATION DATA")
print("="*60)

SQL ANALYSIS ON CUSTOMER SEGMENTATION DATA


In [3]:
# Load your cleaned data
df = pd.read_csv('/content/customer_segmentation_cleaned .csv')
print(f"\n✅ Loaded {len(df):,} customers")
print(f"📋 Columns: {df.columns.tolist()}")


✅ Loaded 5,878 customers
📋 Columns: ['Customer ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment', 'ML_Segment', 'Cluster']


In [4]:
# Create SQL database
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False)
print("✅ SQL database created!\n")

✅ SQL database created!



In [5]:
# ============================================
# QUERY 1: Segment Performance
# ============================================
print("="*50)
print("QUERY 1: SEGMENT PERFORMANCE")
print("="*50)

query1 = """
SELECT
    Segment,
    COUNT(*) as customers,
    ROUND(SUM(Monetary), 2) as total_revenue,
    ROUND(AVG(Monetary), 2) as avg_spend,
    ROUND(AVG(Frequency), 2) as avg_orders,
    ROUND(AVG(Recency), 2) as avg_recency,
    ROUND(100.0 * SUM(Monetary) / (SELECT SUM(Monetary) FROM customers), 1) as revenue_pct
FROM customers
GROUP BY Segment
ORDER BY total_revenue DESC
"""
result1 = pd.read_sql_query(query1, conn)
print(result1.to_string(index=False))

QUERY 1: SEGMENT PERFORMANCE
              Segment  customers  total_revenue  avg_spend  avg_orders  avg_recency  revenue_pct
          👑 Champions       1182    12289134.80   10396.90       18.71        21.62         69.3
           ⚠️ At Risk        880     2221679.29    2524.64        6.53       276.13         12.5
        😴 Hibernating       2045     1135329.16     555.17        1.49       410.56          6.4
📈 Potential Loyalists        540      952765.39    1764.38        5.41        54.92          5.4
      🆕 New Customers        894      769953.17     861.25        1.78        39.72          4.3
    ⭐ Loyal Customers        337      374567.36    1111.48        4.64        12.58          2.1


In [6]:
# ============================================
# QUERY 2: Top 10 Customers (FIXED - using quotes)
# ============================================
print("\n" + "="*50)
print("QUERY 2: TOP 10 CUSTOMERS BY LIFETIME VALUE")
print("="*50)

query2 = """
SELECT
    "Customer ID" as Customer_ID,
    ROUND(Monetary, 2) as total_spent,
    Frequency as orders,
    Recency as days_since_last,
    Segment
FROM customers
ORDER BY Monetary DESC
LIMIT 10
"""
result2 = pd.read_sql_query(query2, conn)
print(result2.to_string(index=False))


QUERY 2: TOP 10 CUSTOMERS BY LIFETIME VALUE
 Customer_ID  total_spent  orders  days_since_last         Segment
     18102.0    608821.65     145                0     👑 Champions
     14646.0    528602.52     151                1     👑 Champions
     14156.0    313946.37     156                9     👑 Champions
     14911.0    295972.63     398                0     👑 Champions
     17450.0    246973.09      51                7     👑 Champions
     13694.0    196482.81     143                3     👑 Champions
     17511.0    175603.55      60                2     👑 Champions
     16446.0    168472.50       2                0 🆕 New Customers
     16684.0    147142.77      55                3     👑 Champions
     12415.0    144458.37      28               23     👑 Champions


In [7]:
# QUERY 3: At Risk Customers (FIXED)
# ============================================
print("\n" + "="*50)
print("QUERY 3: AT RISK CUSTOMERS (Win-back Opportunity)")
print("="*50)

query3 = """
SELECT
    "Customer ID" as Customer_ID,
    ROUND(Monetary, 2) as historical_spend,
    Frequency as orders,
    Recency as days_inactive,
    Segment
FROM customers
WHERE Segment LIKE '%At Risk%'
ORDER BY Monetary DESC
LIMIT 10
"""
result3 = pd.read_sql_query(query3, conn)
print(result3.to_string(index=False))


QUERY 3: AT RISK CUSTOMERS (Win-back Opportunity)
 Customer_ID  historical_spend  orders  days_inactive    Segment
     12346.0          77556.46      12            325 ⚠️ At Risk
     16754.0          67502.47      29            371 ⚠️ At Risk
     17850.0          56600.08     155            371 ⚠️ At Risk
     13093.0          54943.65      55            275 ⚠️ At Risk
     13902.0          34095.26       5            631 ⚠️ At Risk
     13802.0          26259.11      19            138 ⚠️ At Risk
     12482.0          23691.40      29            575 ⚠️ At Risk
     14063.0          22710.20       9            437 ⚠️ At Risk
     15808.0          18409.93      21            305 ⚠️ At Risk
     13027.0          17335.20      19            113 ⚠️ At Risk


In [8]:
# ============================================
# QUERY 4: RFM Score Distribution
# ============================================
print("\n" + "="*50)
print("QUERY 4: RFM SCORE DISTRIBUTION")
print("="*50)

query4 = """
SELECT
    RFM_Score,
    COUNT(*) as customers,
    ROUND(AVG(Monetary), 2) as avg_spend,
    ROUND(SUM(Monetary), 2) as total_revenue
FROM customers
GROUP BY RFM_Score
ORDER BY RFM_Score DESC
LIMIT 10
"""
result4 = pd.read_sql_query(query4, conn)
print(result4.to_string(index=False))


QUERY 4: RFM SCORE DISTRIBUTION
 RFM_Score  customers  avg_spend  total_revenue
       444        661   14567.99     9629444.41
       443        104    1755.38      182559.52
       442          9     764.07        6876.64
       434         72    4121.84      296772.56
       433        208    1396.55      290481.63
       432        113     665.98       75256.11
       431          7     279.00        1952.98
       424          9   21523.60      193712.40
       423         63    1256.27       79145.16
       422        120     576.47       69175.86


In [9]:
# ============================================
# QUERY 5: Customer Tiers
# ============================================
print("\n" + "="*50)
print("QUERY 5: CUSTOMER TIERS")
print("="*50)

query5 = """
SELECT
    CASE
        WHEN Monetary >= 10000 THEN '👑 Platinum'
        WHEN Monetary >= 5000 THEN '🥇 Gold'
        WHEN Monetary >= 1000 THEN '🥈 Silver'
        ELSE '🥉 Bronze'
    END as tier,
    COUNT(*) as customers,
    ROUND(SUM(Monetary), 2) as total_revenue,
    ROUND(AVG(Monetary), 2) as avg_spend,
    ROUND(100.0 * SUM(Monetary) / (SELECT SUM(Monetary) FROM customers), 1) as revenue_pct
FROM customers
GROUP BY tier
ORDER BY total_revenue DESC
"""
result5 = pd.read_sql_query(query5, conn)
print(result5.to_string(index=False))



QUERY 5: CUSTOMER TIERS
      tier  customers  total_revenue  avg_spend  revenue_pct
👑 Platinum        267     8958557.75   33552.65         50.5
  🥈 Silver       2077     4647860.73    2237.78         26.2
    🥇 Gold        403     2815665.68    6986.76         15.9
  🥉 Bronze       3131     1321345.01     422.02          7.4


In [10]:
# ============================================
# QUERY 6: Recency Status
# ============================================
print("\n" + "="*50)
print("QUERY 6: CUSTOMER RECENCY STATUS")
print("="*50)

query6 = """
SELECT
    CASE
        WHEN Recency <= 30 THEN '🟢 Active (0-30 days)'
        WHEN Recency <= 90 THEN '🟡 Moderate (31-90 days)'
        WHEN Recency <= 180 THEN '🟠 Dormant (91-180 days)'
        ELSE '🔴 Inactive (180+ days)'
    END as status,
    COUNT(*) as customers,
    ROUND(SUM(Monetary), 2) as revenue,
    ROUND(100.0 * SUM(Monetary) / (SELECT SUM(Monetary) FROM customers), 1) as revenue_pct
FROM customers
GROUP BY status
ORDER BY
    CASE status
        WHEN '🟢 Active (0-30 days)' THEN 1
        WHEN '🟡 Moderate (31-90 days)' THEN 2
        WHEN '🟠 Dormant (91-180 days)' THEN 3
        ELSE 4
    END
"""
result6 = pd.read_sql_query(query6, conn)
print(result6.to_string(index=False))



QUERY 6: CUSTOMER RECENCY STATUS
                 status  customers     revenue  revenue_pct
   🟢 Active (0-30 days)       1694 11380743.90         64.1
🟡 Moderate (31-90 days)       1199  2900457.93         16.3
🟠 Dormant (91-180 days)        587  1068784.28          6.0
 🔴 Inactive (180+ days)       2398  2393443.07         13.5


In [11]:
# ============================================
# QUERY 7: Champions vs Hibernating
# ============================================
print("\n" + "="*50)
print("QUERY 7: CHAMPIONS VS HIBERNATING")
print("="*50)

query7 = """
SELECT
    Segment,
    COUNT(*) as customers,
    ROUND(AVG(Monetary), 2) as avg_spend,
    ROUND(AVG(Frequency), 2) as avg_orders,
    ROUND(AVG(Recency), 2) as avg_days_inactive,
    MIN(Recency) as most_recent,
    MAX(Recency) as least_recent
FROM customers
WHERE Segment IN ('👑 Champions', '😴 Hibernating')
GROUP BY Segment
"""
result7 = pd.read_sql_query(query7, conn)
print(result7.to_string(index=False))


QUERY 7: CHAMPIONS VS HIBERNATING
      Segment  customers  avg_spend  avg_orders  avg_days_inactive  most_recent  least_recent
  👑 Champions       1182   10396.90       18.71              21.62            0            95
😴 Hibernating       2045     555.17        1.49             410.56           96           738


In [12]:
# ============================================
# QUERY 8: ML Cluster Analysis
# ============================================
print("\n" + "="*50)
print("QUERY 8: MACHINE LEARNING CLUSTER ANALYSIS")
print("="*50)

query8 = """
SELECT
    ML_Segment as cluster,
    COUNT(*) as customers,
    ROUND(AVG(Monetary), 2) as avg_spend,
    ROUND(AVG(Frequency), 2) as avg_frequency,
    ROUND(AVG(Recency), 2) as avg_recency,
    ROUND(100.0 * SUM(Monetary) / (SELECT SUM(Monetary) FROM customers), 1) as revenue_pct
FROM customers
GROUP BY ML_Segment
ORDER BY avg_spend DESC
"""
result8 = pd.read_sql_query(query8, conn)
print(result8.to_string(index=False))



QUERY 8: MACHINE LEARNING CLUSTER ANALYSIS
     cluster  customers  avg_spend  avg_frequency  avg_recency  revenue_pct
  High Value       1470    9850.43          16.60        69.61         81.6
Medium Value       1469    1445.58           4.77       155.23         12.0
   Low Value       1761     433.37           2.17       131.13          4.3
    Inactive       1178     319.69           1.47       523.15          2.1


In [13]:
# ============================================
# QUERY 9: Pareto Analysis
# ============================================
print("\n" + "="*50)
print("QUERY 9: PARETO ANALYSIS (80/20 RULE)")
print("="*50)

query9 = """
WITH ranked AS (
    SELECT
        Monetary,
        SUM(Monetary) OVER (ORDER BY Monetary DESC) as running_total,
        SUM(Monetary) OVER () as total,
        ROW_NUMBER() OVER (ORDER BY Monetary DESC) as row_num,
        COUNT(*) OVER () as total_rows
    FROM customers
)
SELECT
    ROUND(100.0 * row_num / total_rows, 1) as customer_pct,
    ROUND(100.0 * running_total / total, 1) as revenue_pct
FROM ranked
WHERE row_num = CAST(total_rows * 0.2 AS INTEGER)
LIMIT 1
"""
result9 = pd.read_sql_query(query9, conn)
if len(result9) > 0:
    print(f"📈 Top {result9['customer_pct'].iloc[0]}% of customers generate {result9['revenue_pct'].iloc[0]}% of revenue")
else:
    print("   Calculation complete")


QUERY 9: PARETO ANALYSIS (80/20 RULE)
📈 Top 20.0% of customers generate 77.2% of revenue


In [14]:
# ============================================
# QUERY 10: Cross-Sell Opportunities
# ============================================
print("\n" + "="*50)
print("QUERY 10: CROSS-SELL OPPORTUNITIES")
print("="*50)

query10 = """
SELECT
    Segment,
    COUNT(*) as customers,
    ROUND(AVG(Frequency), 2) as avg_orders,
    ROUND(AVG(Monetary), 2) as avg_spend,
    CASE
        WHEN AVG(Frequency) < 5 AND AVG(Monetary) > 2000 THEN '💰 Upsell Opportunity'
        WHEN AVG(Frequency) > 10 AND AVG(Monetary) < 1000 THEN '🛒 Cross-sell Opportunity'
        WHEN AVG(Recency) > 180 AND AVG(Monetary) > 1000 THEN '🎯 Win-back Opportunity'
        ELSE '📈 Standard'
    END as recommendation
FROM customers
GROUP BY Segment
"""
result10 = pd.read_sql_query(query10, conn)
print(result10.to_string(index=False))


QUERY 10: CROSS-SELL OPPORTUNITIES
              Segment  customers  avg_orders  avg_spend         recommendation
           ⚠️ At Risk        880        6.53    2524.64 🎯 Win-back Opportunity
    ⭐ Loyal Customers        337        4.64    1111.48             📈 Standard
      🆕 New Customers        894        1.78     861.25             📈 Standard
          👑 Champions       1182       18.71   10396.90             📈 Standard
📈 Potential Loyalists        540        5.41    1764.38             📈 Standard
        😴 Hibernating       2045        1.49     555.17             📈 Standard


In [15]:
# ============================================
# EXPORT RESULTS FOR POWER BI
# ============================================
print("\n" + "="*50)
print("EXPORTING RESULTS FOR POWER BI")
print("="*50)

# Save all results to CSV
result1.to_csv('sql_segment_performance.csv', index=False)
result2.to_csv('sql_top_customers.csv', index=False)
result5.to_csv('sql_customer_tiers.csv', index=False)
result6.to_csv('sql_recency_status.csv', index=False)
result8.to_csv('sql_cluster_analysis.csv', index=False)

print("✅ Exported files for Power BI:")
print("   - sql_segment_performance.csv")
print("   - sql_top_customers.csv")
print("   - sql_customer_tiers.csv")
print("   - sql_recency_status.csv")
print("   - sql_cluster_analysis.csv")

# Close connection
conn.close()

print("\n" + "="*60)
print("✅ SQL ANALYSIS COMPLETE!")
print("="*60)


EXPORTING RESULTS FOR POWER BI
✅ Exported files for Power BI:
   - sql_segment_performance.csv
   - sql_top_customers.csv
   - sql_customer_tiers.csv
   - sql_recency_status.csv
   - sql_cluster_analysis.csv

✅ SQL ANALYSIS COMPLETE!
